# SpeechLM API in vLLM example

In [ ]:
# example of llm streaming usage

from operator import ne
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

engine_args = AsyncEngineArgs(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_model_len=256,
    gpu_memory_utilization=0.8,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=20, temperature=0.7, top_p=0.9)

tokens = engine.tokenizer.encode("My name is")
inputs = {
    "prompt_token_ids": [0] * len(tokens),
    "custom_inputs": {
        "custom_tokens": torch.tensor(tokens, dtype=torch.int32),
    }
}

sampled_tokens = []
async for output in engine.generate(
    inputs,
    sampling_params=sampling_params,
    request_id="1",
):
    new_token = output.outputs[0].token_ids[-1]
    sampled_tokens.append(new_token)
    if not output.finished:
        await engine.append_request(
            request_id="1",
            custom_inputs={"custom_tokens": torch.tensor([new_token], dtype=torch.int32)}
        )

print(engine.tokenizer.decode(sampled_tokens))
